# Colab smoke test 3: the Stop button during Run all

Use **Runtime > Run all**. While Test 1 is running (it takes 60 seconds; open its Answer to watch it, or just wait
10 seconds), press **Runtime > Interrupt execution** (or the stop square). Expected: Run all stops, so the
cell under **End** does **not** show `END`.

In [ ]:
import sys, IPython, ipykernel
print("Python", sys.version.split()[0], "| IPython", IPython.__version__, "| ipykernel", ipykernel.__version__)

from os.path import exists
from urllib.request import urlretrieve
if not exists('jupyturtle.py'):
    urlretrieve('https://github.com/ramalho/jupyturtle/releases/download/2024-03/jupyturtle.py', 'jupyturtle.py')
from jupyturtle import make_turtle, forward, left


def run_code(code):
    """A: run code (a string) as if it were a cell of its own; an error is shown as a normal
    (red) error output, but the cell itself succeeds, so "Run all" goes on."""
    try:
        ip = get_ipython()
    except NameError:              # plain Python: just run it
        exec(code, globals())
        return
    result = ip.run_cell(code, store_history=False)
    if isinstance(result.error_in_exec, KeyboardInterrupt):
        raise KeyboardInterrupt    # the Stop button still stops "Run all"


def run_code_stderr(code):
    """B (fallback 1): the same, but the traceback is printed on stderr, so there is no
    'error' output at all."""
    ip = get_ipython()
    def to_stderr(etype, evalue, stb):
        print(ip.InteractiveTB.stb2text(stb), file=sys.stderr)
    ip._showtraceback = to_stderr
    try:
        result = ip.run_cell(code, store_history=False)
    finally:
        del ip._showtraceback
    if isinstance(result.error_in_exec, KeyboardInterrupt):
        raise KeyboardInterrupt


def run_hidden(code):
    """C (fallback 2): run code, capture everything it shows (prints, errors, last expression,
    turtle drawing) and show it inside a closed HTML <details> ("Show the output"), so the cell
    can stay visible without giving the answer away."""
    import html, re
    from IPython.utils.capture import capture_output
    from IPython.display import display, HTML
    ip = get_ipython()
    def to_stderr(etype, evalue, stb):
        print(ip.InteractiveTB.stb2text(stb), file=sys.stderr)
    ip._showtraceback = to_stderr
    try:
        with capture_output() as cap:
            result = ip.run_cell(code, store_history=False)
    finally:
        del ip._showtraceback
    ansi = re.compile(r'\x1b\[[0-9;]*m')
    parts = []
    if cap.stdout:
        parts.append('<pre>' + html.escape(cap.stdout) + '</pre>')
    rich = {}
    for k, o in enumerate(cap.outputs):          # keep the last version of each display (turtle updates)
        key = (o.transient or {}).get('display_id', k) if hasattr(o, 'transient') else k
        rich[key] = o.data
    for data in rich.values():
        if 'text/html' in data:
            parts.append(data['text/html'])
        elif 'text/plain' in data:
            parts.append('<pre>' + html.escape(data['text/plain']) + '</pre>')
    if cap.stderr:
        parts.append('<pre style="color:#c00">' + html.escape(ansi.sub('', cap.stderr)) + '</pre>')
    display(HTML('<details><summary><b>Show the output</b></summary>' + ''.join(parts) + '</details>'))
    if isinstance(result.error_in_exec, KeyboardInterrupt):
        raise KeyboardInterrupt


### Test 1: a slow cell run by run_code

```python
import time
for i in range(60):
    time.sleep(1)
print('finished')
```

#### Answer

```text
finished
```

In [ ]:
run_code("""
import time
for i in range(60):
    time.sleep(1)
print('finished')
""")

## End

In [ ]:
print('END: Run all was NOT stopped')